# RL Masterclass 02: Q-Learning & Temporal Difference Methods

**Track:** Reinforcement Learning (0 to Masterclass)  
**Prerequisites:** RL 01 (MDP Foundations)  
**Difficulty:** ⭐⭐ Intermediate  
**Estimated Time:** 90–120 minutes

---

## 🎓 Socratic Deep Check

> *"Q-Learning learns WITHOUT a model of the environment — it doesn't know the transition probabilities P(s'|s,a). It only learns from experience. How can trial-and-error converge to the OPTIMAL policy when the agent doesn't even know the rules of the game?"*

---

## From Dynamic Programming to Learning

In RL 01, Value Iteration needed the FULL model: all states, all transitions, all rewards. But in real problems:

| Approach | Needs Model? | Needs All States? | Scalable? |
|----------|:-:|:-:|:-:|
| Dynamic Programming (RL/01) | ✅ Yes | ✅ Yes | ❌ No |
| Monte Carlo | ❌ No | ❌ No | ⚠️ Slow |
| **Temporal Difference (TD)** | ❌ No | ❌ No | ✅ Yes |
| **Q-Learning** | ❌ No | ❌ No | ✅ Yes |

## Table of Contents
1. Monte Carlo vs Temporal Difference
2. Q-Values: Action-Value Function
3. Q-Learning Algorithm
4. Exploration vs Exploitation (ε-greedy)

---

### 🎓 Junior to Senior: Concept Bridge

**The Senior Perspective:** Q-Learning is the most important classical RL algorithm — it's the foundation that DQN (RL/03), Actor-Critic (RL/04), and PPO (RL/05) all build upon. Once you understand Q-values and the TD update rule, everything else is an extension.

**Mental Model:** Imagine learning to drive in a new city. You don't have a map (no model). You try different routes (exploration), and over time you learn which turns are good (high Q-value) and which lead to dead ends (low Q-value). Eventually, you know the best route from any intersection to your destination — that's Q-learning.

**Common Junior Pitfall:** Setting exploration rate (ε) too low too fast. The agent finds a "pretty good" policy early and sticks with it forever, missing the OPTIMAL policy. Always decay ε slowly and never set it to exactly 0.

---

In [ ]:
!pip install -q numpy matplotlib

## 1 · Monte Carlo vs Temporal Difference

### Two Ways to Learn Without a Model

**Monte Carlo:** Wait until the episode ENDS, then update values based on the actual return.

$$V(s_t) \leftarrow V(s_t) + \alpha \left[ \underbrace{G_t}_{\text{actual return}} - V(s_t) \right]$$

**Temporal Difference (TD):** Update IMMEDIATELY after each step using an ESTIMATE of the return.

$$V(s_t) \leftarrow V(s_t) + \alpha \left[ \underbrace{r_{t+1} + \gamma V(s_{t+1})}_{\text{estimated return (TD target)}} - V(s_t) \right]$$

| | Monte Carlo | Temporal Difference |
|---|---|---|
| Updates when? | End of episode | Every step |
| Needs episodes to end? | Yes | No |
| Variance | High (full returns are noisy) | Low (single-step estimates) |
| Bias | Unbiased | Biased (uses estimated V) |
| Best for | Short episodes | Continuing tasks |

**Key insight:** The TD update $r + \gamma V(s')$ is called "bootstrapping" — using your own estimate to update yourself. This is the same idea as the Bellman equation, but learned from experience instead of computed from the model.

In [ ]:
import numpy as np
import matplotlib.pyplot as plt

# GridWorld from RL/01
class GridWorld:
    def __init__(self, size=4):
        self.size = size
        self.goal = (size-1, size-1)
        self.reset()
    def reset(self):
        self.pos = (0, 0)
        return self.pos
    def step(self, action):
        moves = {0: (-1,0), 1: (0,1), 2: (1,0), 3: (0,-1)}
        dx, dy = moves[action]
        new_x = max(0, min(self.size-1, self.pos[0] + dx))
        new_y = max(0, min(self.size-1, self.pos[1] + dy))
        self.pos = (new_x, new_y)
        if self.pos == self.goal:
            return self.pos, +10, True
        return self.pos, -1, False

# Compare MC vs TD learning on GridWorld
def td_value_learning(env, episodes=500, alpha=0.1, gamma=0.9, epsilon=0.1):
    """Learn V(s) using Temporal Difference (TD(0))."""
    V = np.zeros((env.size, env.size))
    history = []
    
    for ep in range(episodes):
        state = env.reset()
        total_reward = 0
        
        for _ in range(100):
            action = np.random.randint(4)  # Random policy
            next_state, reward, done = env.step(action)
            total_reward += reward
            
            # TD Update: V(s) += α * [r + γV(s') - V(s)]
            td_target = reward + gamma * V[next_state] * (1 - done)
            td_error = td_target - V[state]
            V[state] = V[state] + alpha * td_error
            
            state = next_state
            if done:
                break
        
        history.append(total_reward)
    
    return V, history

env = GridWorld(4)
V_td, rewards_td = td_value_learning(env, episodes=1000)

print('TD(0) Learned Value Function:')
print(np.round(V_td, 1))
print(f'\nValues increase toward goal (3,3) — the agent has learned!')
print(f'Compare to Value Iteration (RL/01) — similar pattern, learned from experience!')

---
## 2 · Q-Values: Rating Actions, Not Just States

V(s) tells you how good a STATE is. But to ACT, you need to know how good each ACTION is in that state.

$$Q(s, a) = \text{"How good is it to take action } a \text{ in state } s \text{?"}$$

The relationship:
$$V(s) = \max_a Q(s, a)$$

If you know Q-values, the optimal policy is trivial:
$$\pi^*(s) = \arg\max_a Q(s, a) \quad \text{(just pick the action with highest Q)}$$

This is why Q-learning is so powerful — learn the Q-table, and the policy falls out for free.

---
## 3 · Q-Learning Algorithm

### The Update Rule

$$Q(s, a) \leftarrow Q(s, a) + \alpha \left[ \underbrace{r + \gamma \max_{a'} Q(s', a')}_{\text{TD target}} - Q(s, a) \right]$$

In plain English:
1. Take action $a$ in state $s$
2. Get reward $r$ and arrive at state $s'$
3. **TD target** = reward I got + best I could do from $s'$
4. **TD error** = what I expected minus what I got
5. Nudge Q(s,a) toward the target by learning rate $\alpha$

### Key Property: Off-Policy

Q-learning uses $\max_{a'} Q(s', a')$ — the BEST possible action from $s'$. This is the optimal action, regardless of what the agent actually does (which includes random exploration). This makes Q-learning **off-policy**: it learns the optimal policy WHILE following an exploratory policy.

In [ ]:
import numpy as np
import matplotlib.pyplot as plt

class QLearningAgent:
    """Q-Learning from scratch.
    
    This is the algorithm that started deep RL.
    When you replace the Q-table with a neural network,
    you get DQN (RL/03) — the algorithm that beat Atari.
    """
    
    def __init__(self, state_space, n_actions, alpha=0.1, gamma=0.99, epsilon=1.0):
        self.Q = {}  # Q-table: {state: [Q(s,a) for each action]}
        self.n_actions = n_actions
        self.alpha = alpha      # Learning rate
        self.gamma = gamma      # Discount factor
        self.epsilon = epsilon  # Exploration rate
    
    def get_q(self, state):
        """Get Q-values for a state (initialize to 0 if unseen)."""
        if state not in self.Q:
            self.Q[state] = np.zeros(self.n_actions)
        return self.Q[state]
    
    def choose_action(self, state):
        """ε-greedy: explore with probability ε, exploit otherwise."""
        if np.random.random() < self.epsilon:
            return np.random.randint(self.n_actions)  # Random action
        return np.argmax(self.get_q(state))           # Best known action
    
    def learn(self, state, action, reward, next_state, done):
        """The Q-Learning update rule."""
        q_current = self.get_q(state)[action]
        q_next_max = 0 if done else np.max(self.get_q(next_state))
        
        # TD target and error
        td_target = reward + self.gamma * q_next_max
        td_error = td_target - q_current
        
        # Update Q-value
        self.Q[state][action] += self.alpha * td_error

# Train Q-Learning on GridWorld
env = GridWorld(4)
agent = QLearningAgent(state_space=16, n_actions=4,
                       alpha=0.1, gamma=0.99, epsilon=1.0)

rewards_per_episode = []
epsilon_history = []

for episode in range(2000):
    state = env.reset()
    total_reward = 0
    
    for step in range(100):
        action = agent.choose_action(state)
        next_state, reward, done = env.step(action)
        agent.learn(state, action, reward, next_state, done)
        total_reward += reward
        state = next_state
        if done:
            break
    
    rewards_per_episode.append(total_reward)
    epsilon_history.append(agent.epsilon)
    
    # Decay exploration: start random, gradually exploit
    agent.epsilon = max(0.01, agent.epsilon * 0.998)

# Results
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# Learning curve
window = 50
smoothed = [np.mean(rewards_per_episode[max(0,i-window):i+1]) for i in range(len(rewards_per_episode))]
axes[0].plot(smoothed, lw=1.5, color='steelblue')
axes[0].set_xlabel('Episode')
axes[0].set_ylabel('Total Reward (smoothed)')
axes[0].set_title('Q-Learning: Learning Curve')
axes[0].axhline(y=4, color='green', linestyle='--', label='Optimal (reward=4)')
axes[0].legend()

# Epsilon decay
axes[1].plot(epsilon_history, lw=1.5, color='coral')
axes[1].set_xlabel('Episode')
axes[1].set_ylabel('ε (exploration rate)')
axes[1].set_title('Exploration → Exploitation')

plt.tight_layout()
plt.savefig('q_learning.png', dpi=100)
plt.show()

# Show learned policy
arrows = {0: '↑', 1: '→', 2: '↓', 3: '←'}
print('Learned Policy:')
for i in range(4):
    row = ''
    for j in range(4):
        if (i,j) == (3,3):
            row += ' ★ '
        else:
            q = agent.get_q((i,j))
            best = arrows[np.argmax(q)]
            row += f' {best} '
    print(f'  {row}')
print(f'\nQ-learning found the optimal path WITHOUT knowing the rules!')

---
## 4 · Exploration vs Exploitation

### The Fundamental Tradeoff

- **Exploit:** Take the best known action → good short-term reward
- **Explore:** Try random actions → might find something better

If you always exploit, you might miss the optimal path. If you always explore, you never use what you've learned.

### Common Strategies

| Strategy | How It Works | Pros | Cons |
|----------|-------------|------|------|
| **ε-greedy** | Random action with prob ε | Simple, effective | Explores uniformly (even bad actions) |
| **ε-decay** | ε decreases over time | Explores early, exploits late | Need to tune decay rate |
| **Boltzmann** | Sample actions by softmax(Q/τ) | Explores good actions more | Temperature tuning |
| **UCB** | Bonus for unexplored actions | Theoretically optimal | More complex |

### 🎓 DEEP QUESTION ANSWERED

**Q:** *How can trial-and-error converge to the OPTIMAL policy?*

**A:** Two key ingredients:
1. **The Q-learning update converges because of the Robbins-Monro conditions**: as long as every (state, action) pair is visited infinitely often and the learning rate decreases appropriately, Q-values converge to their true values. ε-greedy exploration guarantees every pair is eventually visited.
2. **The max in the update rule**: Q-learning uses max_a' Q(s',a'), which targets the OPTIMAL Q-value, not the value of the current policy. This "off-policy" trick means the agent can explore randomly while learning optimally.

---

## ✅ Knowledge Check

### Q1: On-policy vs off-policy
SARSA uses the actual next action $a'$ in its update: Q(s,a) += α[r + γQ(s',a') - Q(s,a)]. How does this differ from Q-learning?

<details><summary>👉 Answer</summary>

SARSA is ON-policy: it updates toward the value of the action it ACTUALLY took (including exploratory random actions). Q-learning is OFF-policy: it updates toward the BEST possible action (max). SARSA learns a safer policy (accounts for its own randomness), Q-learning learns the optimal policy (assumes perfect execution). SARSA is better for real-time control where you can't afford risky exploration.
</details>

### Q2: Q-table limitations
Why can't you use a Q-table for a game with 10 million possible states?

<details><summary>👉 Answer</summary>

The table would have 10M rows × n_actions columns — too large to store and too slow to fill (you'd need to visit each (state, action) pair many times). Solution: replace the table with a NEURAL NETWORK that approximates Q(s,a). Input: state, output: Q-values for all actions. This is DQN (RL/03). The network generalizes across similar states.
</details>

### Q3: Epsilon decay
What goes wrong if you never decay epsilon (always ε=0.5)?

<details><summary>👉 Answer</summary>

The agent takes random actions 50% of the time FOREVER, even after learning the optimal policy. Evaluation performance would be poor because half the actions are random. You need to decay ε so the agent gradually shifts from exploration to exploitation. Common schedule: ε = max(0.01, ε × 0.995).
</details>

---

## 🔨 Practical Practice

### Exercise 1: Cliff Walking
Create a 4×12 grid where the bottom row (except start and goal) is a "cliff" with reward -100. Compare Q-learning vs SARSA — which finds a safer path?

### Exercise 2: Multi-Armed Bandit
Implement a 10-armed bandit problem (no states, just actions). Compare ε-greedy vs UCB exploration strategies over 1000 steps.

### Exercise 3: Taxi Problem
Implement Q-learning for the Taxi environment (500 states, 6 actions). Track how many episodes are needed to achieve optimal performance.

---

**Next →** RL 03: Deep Q-Networks (DQN)